# Module 3.1 - Agentic RAG

**Reference:** [`01-agentic-rag.md`](./01-agentic-rag.md)

## What you'll do in this notebook

- Use the LLM itself to rewrite a user's query in several different ways (query expansion).
- Break a complex, multi-part question into simpler sub-questions (query decomposition).
- Run retrieval against several query variants and combine the results with deduplication + frequency ranking.
- Decide on the right strategy for a given query.

## Setup

We reuse the `RAGChatbot` pattern from Module 2. Here we set up a small collection so you can focus on the agentic bit.

In [ ]:
import os
import shutil
from collections import Counter
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
EMBED_MODEL = "text-embedding-3-small"
CHROMA_PATH = "./chroma_store_m3_1"

if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

chroma = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma.create_collection("agentic_demo")

def embed_batch(texts):
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in resp.data]

DOCS = [
    "Python was designed by Guido van Rossum and first released in 1991.",
    "Django is a Python web framework written in Python and released in 2005.",
    "Flask is a lightweight Python web framework released in 2010.",
    "Exception handling in Python uses try, except, and finally blocks.",
    "Python context managers are implemented with __enter__ and __exit__ methods.",
    "JavaScript was created by Brendan Eich at Netscape in 1995.",
    "React is a JavaScript library for building user interfaces, released by Facebook in 2013.",
    "Express is a minimal and flexible Node.js web framework.",
    "Error handling in JavaScript uses try, catch, and finally blocks.",
    "Async/await in JavaScript was standardised in ES2017.",
]
METAS = [
    {"source": f"doc_{i}.md", "language": lang}
    for i, lang in enumerate(["python"] * 5 + ["javascript"] * 5)
]

collection.add(
    ids=[f"doc_{i}" for i in range(len(DOCS))],
    embeddings=embed_batch(DOCS),
    documents=DOCS,
    metadatas=METAS,
)
print(f"OK: {collection.count()} documents indexed")

## Exercise 1 - Query expansion

Ask the LLM to rewrite a user question in three different ways. This helps when the user's wording doesn't match the documents (e.g. "handle errors" vs "exception handling").

In [ ]:
def expand_query(original: str, n: int = 3) -> list[str]:
    """Return [original] plus n alternative phrasings."""
    # TODO:
    # 1. Build a prompt that asks the LLM to generate n alternative
    #    phrasings, one per line, with no numbering or commentary.
    # 2. Call client.chat.completions.create with a low-ish temperature (0.5).
    # 3. Split the reply on newlines, strip each line, drop empty ones.
    # 4. Return [original] + the alternatives.
    raise NotImplementedError

variations = expand_query("How do I handle errors in Python?", n=3)
for v in variations:
    print("-", v)

## Exercise 2 - Query decomposition

When a user's question has several parts, expansion just rewrites the whole thing. Decomposition instead breaks it into independent sub-questions, each of which can be retrieved separately.

In [ ]:
def decompose_query(complex_query: str) -> list[str]:
    """Return a list of independent sub-questions."""
    # TODO: prompt the LLM to break the query into sub-questions, one per line,
    # no numbering or prose. Use temperature=0.2 so decomposition is stable.
    raise NotImplementedError

subqs = decompose_query("Compare Python and JavaScript for building web apps.")
for q in subqs:
    print("-", q)

## Exercise 3 - Multi-step retrieval

Run retrieval once per variant, then combine: deduplicate, and rank documents by how often they appear across variants (a doc that shows up for multiple variants is probably more relevant than one that shows up for only one).

In [ ]:
def retrieve(query: str, top_k: int = 3) -> list[tuple[str, str]]:
    """Return (doc_id, document_text) pairs."""
    emb = embed_batch([query])[0]
    res = collection.query(query_embeddings=[emb], n_results=top_k)
    return list(zip(res["ids"][0], res["documents"][0]))

def agentic_retrieve(query: str, strategy: str = "expand", top_k: int = 3) -> list[tuple[str, str, int]]:
    """Return (doc_id, text, frequency) tuples sorted by frequency desc."""
    if strategy == "expand":
        variants = expand_query(query, n=3)
    elif strategy == "decompose":
        variants = decompose_query(query)
    else:
        variants = [query]

    # TODO:
    # 1. For each variant, call retrieve() and collect (doc_id, text) results.
    # 2. Count how many times each doc_id appears across the variants.
    # 3. Deduplicate by doc_id, attach the frequency count, and sort desc by frequency.
    # 4. Return the top_k entries.
    raise NotImplementedError

print("-- expansion --")
for doc_id, text, freq in agentic_retrieve("How do I handle errors in Python?", strategy="expand"):
    print(f"[freq={freq}] ({doc_id}) {text}")

print("\n-- decomposition --")
for doc_id, text, freq in agentic_retrieve("Compare Python and JavaScript for building web apps.", strategy="decompose"):
    print(f"[freq={freq}] ({doc_id}) {text}")

## Reflection

- Which approach worked better for the error-handling query? Why?
- Which approach worked better for the comparison query? Why?
- Each variant costs an extra embedding + vector search. For a real chatbot handling thousands of queries, when would you skip expansion and go straight to single-step retrieval?

## What's next

Expansion and decomposition help you find more candidates. `02-reranking-and-hybrid-search.ipynb` shows how to make the *ranking* of those candidates more precise — first with LLM reranking, then by mixing in a classical keyword search (BM25).